---
title: Piping in Python
subtitle: Chaining Data Transformations with Python's dfply
abstract: Piping is a programming paradigm that allows chaining multiple operations together in a readable and concise manner. It is inspired by the pipe operator (%>%) in R's dplyr package and is implemented in Python using libraries like dfply. This approach is particularly useful for data manipulation tasks, as it enables a clean and logical flow of operations on data.
author:
  - name: Karol Flisikowski
    affiliations: 
      - Gdansk University of Technology
      - Chongqing Technology and Business University
    orcid: 0000-0002-4160-1297
    linkedin: flisik
    email: karol@ctbu.edu.cn
date: 2025-04-28
---

## Piping in Python

Learn how to summarize the columns available in an R data frame. You will also learn how to chain operations together with the pipe operator, and how to compute grouped summaries using.

The dfply package makes it possible to do R's dplyr-style data manipulation with pipes in python on pandas DataFrames.

[dfply website here](https://github.com/kieferk/dfply)

[![](https://www.rforecology.com/pipes_image0.png "https://github.com/kieferk/dfply")](https://github.com/kieferk/dfply)

**Key Features of Piping:**

- Chaining Operations: Use the >> operator to chain multiple operations on a pandas DataFrame.
- Deferred Evaluation: Operations are recorded symbolically and evaluated only when needed.
- Readable Syntax: Makes complex data transformations easier to read and understand.

**Common Functions in dfply:**

- select(): Select specific columns.
- drop(): Drop specific columns.
- filter(): Filter rows based on conditions.
- mutate(): Add or modify columns.
- summarize(): Compute summary statistics.
- group_by(): Group data for aggregation.

Piping simplifies workflows by reducing the need for intermediate variables and making the code more intuitive

In [56]:
import pandas as pd
import polars as pl
import polars.selectors as cs
import seaborn as sns
import pyarrow

cars_pd = sns.load_dataset('mpg')
cars_pl = pl.from_pandas(cars_pd)
#from dfply import *
#cars >> head(3)

In [57]:
# Pandas
cars_pd.head(5)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


In [58]:
# Polars
cars_pl.head(5)

mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
f64,i64,f64,f64,i64,f64,i64,str,str
18.0,8,307.0,130.0,3504,12.0,70,"""usa""","""chevrolet chevelle malibu"""
15.0,8,350.0,165.0,3693,11.5,70,"""usa""","""buick skylark 320"""
18.0,8,318.0,150.0,3436,11.0,70,"""usa""","""plymouth satellite"""
16.0,8,304.0,150.0,3433,12.0,70,"""usa""","""amc rebel sst"""
17.0,8,302.0,140.0,3449,10.5,70,"""usa""","""ford torino"""


## The \>\> and \>\>=

dfply works directly on pandas DataFrames, chaining operations on the data with the >> operator, or alternatively starting with >>= for inplace operations.

*The X DataFrame symbol*

The DataFrame as it is passed through the piping operations is represented by the symbol X. It records the actions you want to take (represented by the Intention class), but does not evaluate them until the appropriate time. Operations on the DataFrame are deferred. Selecting two of the columns, for example, can be done using the symbolic X DataFrame during the piping operations.

### Exercise 1.

Select the columns 'mpg' and 'horsepower' from the cars DataFrame.

In [59]:
# Obsolete dfply / only for reference
# cars
# >> select('mpg','horsepower')
# >> head(5)

In [60]:
# Pandas
cars_pd[['mpg', 'horsepower']].head(5)

,mpg,horsepower
0,18.0,130.0
1,15.0,165.0
2,18.0,150.0
3,16.0,150.0
4,17.0,140.0


In [61]:
# Polars
(cars_pl
 .select(
    pl.col('mpg'),
    pl.col('horsepower'))
.head(5))

mpg,horsepower
f64,f64
18.0,130.0
15.0,165.0
18.0,150.0
16.0,150.0
17.0,140.0


## Selecting and dropping

There are two functions for selection, inverse of each other: select and drop. The select and drop functions accept string labels, integer positions, and/or symbolically represented column names (X.column). They also accept symbolic "selection filter" functions, which will be covered shortly.

### Exercise 2.

Select the columns 'mpg' and 'horsepower' from the cars DataFrame using the drop function.

In [62]:
# Obsolete dfply / only for reference
# cars
# >> drop(X.mpg, X.horsepower)

In [63]:
# Pandas
cars_pd.drop(['mpg', 'horsepower'], axis = 1)

,cylinders,displacement,weight,acceleration,model_year,origin,name
0,8,307.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,8,350.0,3693,11.5,70,usa,buick skylark 320
2,8,318.0,3436,11.0,70,usa,plymouth satellite
3,8,304.0,3433,12.0,70,usa,amc rebel sst
4,8,302.0,3449,10.5,70,usa,ford torino
...,...,...,...,...,...,...,...
393,4,140.0,2790,15.6,82,usa,ford mustang gl
394,4,97.0,2130,24.6,82,europe,vw pickup
395,4,135.0,2295,11.6,82,usa,dodge rampage
396,4,120.0,2625,18.6,82,usa,ford ranger


In [64]:
# Polars
(cars_pl
 .select(
    pl.exclude('mpg', 'horsepower'))
)

cylinders,displacement,weight,acceleration,model_year,origin,name
i64,f64,i64,f64,i64,str,str
8,307.0,3504,12.0,70,"""usa""","""chevrolet chevelle malibu"""
8,350.0,3693,11.5,70,"""usa""","""buick skylark 320"""
8,318.0,3436,11.0,70,"""usa""","""plymouth satellite"""
8,304.0,3433,12.0,70,"""usa""","""amc rebel sst"""
8,302.0,3449,10.5,70,"""usa""","""ford torino"""
…,…,…,…,…,…,…
4,140.0,2790,15.6,82,"""usa""","""ford mustang gl"""
4,97.0,2130,24.6,82,"""europe""","""vw pickup"""
4,135.0,2295,11.6,82,"""usa""","""dodge rampage"""


## Selection using \~

One particularly nice thing about dplyr's selection functions is that you can drop columns inside of a select statement by putting a subtraction sign in front, like so: ... %>% select(-col). The same can be done in dfply, but instead of the subtraction operator you use the tilde ~.

### Exercise 3.

Select all columns except 'model_year', and 'name' from the cars DataFrame.

In [65]:
# Obsolete dfply / only for reference
# cars
# >> select(~X.model_year, ~X.name)

In [66]:
# Pandas
cars_pd.drop(['model_year', 'name'], axis = 1)


,mpg,cylinders,displacement,horsepower,weight,acceleration,origin
0,18.0,8,307.0,130.0,3504,12.0,usa
1,15.0,8,350.0,165.0,3693,11.5,usa
2,18.0,8,318.0,150.0,3436,11.0,usa
3,16.0,8,304.0,150.0,3433,12.0,usa
4,17.0,8,302.0,140.0,3449,10.5,usa
...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790,15.6,usa
394,44.0,4,97.0,52.0,2130,24.6,europe
395,32.0,4,135.0,84.0,2295,11.6,usa
396,28.0,4,120.0,79.0,2625,18.6,usa


In [67]:
# Polars
(cars_pl
 .select(
    pl.exclude('model_year', 'name'))
)

mpg,cylinders,displacement,horsepower,weight,acceleration,origin
f64,i64,f64,f64,i64,f64,str
18.0,8,307.0,130.0,3504,12.0,"""usa"""
15.0,8,350.0,165.0,3693,11.5,"""usa"""
18.0,8,318.0,150.0,3436,11.0,"""usa"""
16.0,8,304.0,150.0,3433,12.0,"""usa"""
17.0,8,302.0,140.0,3449,10.5,"""usa"""
…,…,…,…,…,…,…
27.0,4,140.0,86.0,2790,15.6,"""usa"""
44.0,4,97.0,52.0,2130,24.6,"""europe"""
32.0,4,135.0,84.0,2295,11.6,"""usa"""


## Filtering columns

The vanilla select and drop functions are useful, but there are a variety of selection functions inspired by dplyr available to make selecting and dropping columns a breeze. These functions are intended to be put inside of the select and drop functions, and can be paired with the ~ inverter.

First, a quick rundown of the available functions:

-   starts_with(prefix): find columns that start with a string prefix.
-   ends_with(suffix): find columns that end with a string suffix.
-   contains(substr): find columns that contain a substring in their name.
-   everything(): all columns.
-   columns_between(start_col, end_col, inclusive=True): find columns between a specified start and end column. The inclusive boolean keyword argument indicates whether the end column should be included or not.
-   columns_to(end_col, inclusive=True): get columns up to a specified end column. The inclusive argument indicates whether the ending column should be included or not.
-   columns_from(start_col): get the columns starting at a specified column.

### Exercise 4.

The selection filter functions are best explained by example. Let's say I wanted to select only the columns that started with a "c":

In [68]:
# Obsolete dfply / only for reference
# cars
# >> select(starts_with('c))

In [69]:
# Pandas
cars_pd.filter(regex = "^c")

,cylinders
0,8
1,8
2,8
3,8
4,8
...,...
393,4
394,4
395,4
396,4


In [70]:
# Polars
(cars_pl
 .select(
    cs.starts_with('c'))
)

cylinders
i64
8
8
8
8
8
…
4
4
4


### Exercise 5.

Select the columns that contain the substring "e" from the cars DataFrame.

In [71]:
# Obsolete dfply / only for reference
# cars
# >> select(constain('e')

In [72]:
# Pandas
cars_pd.filter(like = 'e')

,cylinders,displacement,horsepower,weight,acceleration,model_year,name
0,8,307.0,130.0,3504,12.0,70,chevrolet chevelle malibu
1,8,350.0,165.0,3693,11.5,70,buick skylark 320
2,8,318.0,150.0,3436,11.0,70,plymouth satellite
3,8,304.0,150.0,3433,12.0,70,amc rebel sst
4,8,302.0,140.0,3449,10.5,70,ford torino
...,...,...,...,...,...,...,...
393,4,140.0,86.0,2790,15.6,82,ford mustang gl
394,4,97.0,52.0,2130,24.6,82,vw pickup
395,4,135.0,84.0,2295,11.6,82,dodge rampage
396,4,120.0,79.0,2625,18.6,82,ford ranger


In [73]:
# Polars
(cars_pl
 .select(
    cs.contains('e'))
)

cylinders,displacement,horsepower,weight,acceleration,model_year,name
i64,f64,f64,i64,f64,i64,str
8,307.0,130.0,3504,12.0,70,"""chevrolet chevelle malibu"""
8,350.0,165.0,3693,11.5,70,"""buick skylark 320"""
8,318.0,150.0,3436,11.0,70,"""plymouth satellite"""
8,304.0,150.0,3433,12.0,70,"""amc rebel sst"""
8,302.0,140.0,3449,10.5,70,"""ford torino"""
…,…,…,…,…,…,…
4,140.0,86.0,2790,15.6,82,"""ford mustang gl"""
4,97.0,52.0,2130,24.6,82,"""vw pickup"""
4,135.0,84.0,2295,11.6,82,"""dodge rampage"""


### Exercise 6.

Select the columns that are between 'mpg' and 'origin' from the cars DataFrame.

In [74]:
# Obsolete dfply / only for reference
# cars
# >> select(columns_between('mpg', 'origin'))

In [75]:
# Pandas
cars_pd.loc[:, 'mpg':'origin']

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa
...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790,15.6,82,usa
394,44.0,4,97.0,52.0,2130,24.6,82,europe
395,32.0,4,135.0,84.0,2295,11.6,82,usa
396,28.0,4,120.0,79.0,2625,18.6,82,usa


In [76]:
# Polars
# Polars does not support that kind of syntax. Depending on engine used, columns might appear in different order, making operation absurd

## Subsetting and filtering

### row_slice()

Slices of rows can be selected with the row_slice() function. You can pass single integer indices or a list of indices to select rows as with. This is going to be the same as using pandas' .iloc.

#### Exercise 7.

Select the first three rows from the cars DataFrame.

In [77]:
# Obsolete dfply / only for reference
# cars >> row_slice([0, 1, 2])

In [78]:
# Pandas
cars_pd.iloc[0:3]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite


In [79]:
# Polars
(cars_pl
 .head(3))

mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
f64,i64,f64,f64,i64,f64,i64,str,str
18.0,8,307.0,130.0,3504,12.0,70,"""usa""","""chevrolet chevelle malibu"""
15.0,8,350.0,165.0,3693,11.5,70,"""usa""","""buick skylark 320"""
18.0,8,318.0,150.0,3436,11.0,70,"""usa""","""plymouth satellite"""


### distinct()

Selection of unique rows is done with distinct(), which similarly passes arguments and keyword arguments through to the DataFrame's .drop_duplicates() method.

#### Exercise 8.

Select the unique rows from the 'origin' column in the cars DataFrame.

In [80]:
# Obsolete dfply / only for reference
# cars
# >> select('origin')
# >> distinct()

In [81]:
# Pandas
cars_pd['origin'].drop_duplicates()

0        usa
14     japan
19    europe
Name: origin, dtype: str

In [82]:
# Polars
(cars_pl
 .select(
    pl.col('origin'))
 .unique())

origin
str
"""japan"""
"""europe"""
"""usa"""


## mask()

Filtering rows with logical criteria is done with mask(), which accepts boolean arrays "masking out" False labeled rows and keeping True labeled rows. These are best created with logical statements on symbolic Series objects as shown below. Multiple criteria can be supplied as arguments and their intersection will be used as the mask.

### Exercise 9.

Filter the cars DataFrame to only include rows where the 'mpg' is greater than 20, origin Japan, and display the first three rows:

In [98]:
# Obsolete dfply / only for reference
# cars
# >> mask(X.mpg > 20, X.origin == 'japan')
# >> head(3)

In [99]:
# Pandas
cars_pd[(cars_pd['mpg'] > 20) & (cars_pd['origin'] == 'japan')].head(3)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
14,24.0,4,113.0,95.0,2372,15.0,70,japan,toyota corona mark ii
18,27.0,4,97.0,88.0,2130,14.5,70,japan,datsun pl510
29,27.0,4,97.0,88.0,2130,14.5,71,japan,datsun pl510


In [100]:
# Polars
(cars_pl
 .filter(
    (pl.col('mpg') > 20)
    & (pl.col('origin') == 'japan'))
 .head(3))

mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
f64,i64,f64,f64,i64,f64,i64,str,str
24.0,4,113.0,95.0,2372,15.0,70,"""japan""","""toyota corona mark ii"""
27.0,4,97.0,88.0,2130,14.5,70,"""japan""","""datsun pl510"""
27.0,4,97.0,88.0,2130,14.5,71,"""japan""","""datsun pl510"""


## pull()

The pull() function is used to extract a single column from a DataFrame as a pandas Series. This is useful for passing a single column to a function or for further manipulation.

### Exercise 10.

Extract the 'mpg' column from the cars DataFrame, japanese origin, model year 70s, and display the first three rows.

In [86]:
# Obsolete dfply / only for reference
# cars
# >> mask(X.origin == 'Japan', X.model_year >= 70, X.model_year <= 79)
# >> pull('mpg')
# >> head(3)

In [102]:
# Pandas
cars_pd[(cars_pd['origin'] == 'japan') & (cars_pd['model_year'] >= 70) & (cars_pd['model_year'] < 80)]['mpg'][0:3]

14    24.0
18    27.0
29    27.0
Name: mpg, dtype: float64

In [104]:
# Polars
(cars_pl
 .filter(
    (pl.col('origin') == 'japan')
    & (pl.col('model_year').is_between(70, 79)))
 .get_column('mpg')
 .head(3))

mpg
f64
24.0
27.0
27.0


## DataFrame transformation

*mutate()*

The mutate() function is used to create new columns or modify existing columns. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 11.

Create a new column 'mpg_per_cylinder' in the cars DataFrame that is the result of dividing the 'mpg' column by the 'cylinders' column.

In [103]:
# Obsolete dfply / only for reference


*transmute()*

The transmute() function is a combination of a mutate and a selection of the created variables.

### Exercise 12.

Create a new column 'mpg_per_cylinder' in the cars DataFrame that is the result of dividing the 'mpg' column by the 'cylinders' column, and display only the new column.

In [88]:
# Obsolete dfply / only for reference

## Grouping

*group_by() and ungroup()*

The group_by() function is used to group the DataFrame by one or more columns. This is useful for creating groups of rows that can be summarized or transformed together. The ungroup() function is used to remove the grouping.

### Exercise 13.

Group the cars DataFrame by the 'origin' column and calculate the lead of the 'mpg' column.

In [89]:
# Obsolete dfply / only for reference

## Reshaping

*arrange()*

The arrange() function is used to sort the DataFrame by one or more columns. This is useful for reordering the rows of the DataFrame.

### Exercise 14.

Sort the cars DataFrame by the 'mpg' column in descending order.

In [90]:
# Obsolete dfply / only for reference


*rename()*

The rename() function is used to rename columns in the DataFrame. It accepts keyword arguments of the form new_column_name = old_column_name.

### Exercise 15.

Rename the 'mpg' column to 'miles_per_gallon' in the cars DataFrame.

In [91]:
# Obsolete dfply / only for reference


*gather()*

The gather() function is used to reshape the DataFrame from wide to long format. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 16.

Reshape the cars DataFrame from wide to long format by gathering the columns 'mpg', 'horsepower', 'weight', 'acceleration', and 'displacement' into a new column 'variable' and their values into a new column 'value'.

In [92]:
# Obsolete dfply / only for reference


*spread()*

Likewise, you can transform a "long" DataFrame into a "wide" format with the spread(key, values) function. Converting the previously created elongated DataFrame for example would be done like so.

### Exercise 17.

Reshape the cars DataFrame from long to wide format by spreading the 'variable' column into columns and their values into the 'value' column.

In [93]:
# Obsolete dfply / only for reference


## Summarization

*summarize()*

The summarize() function is used to calculate summary statistics for groups of rows. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 18.

Calculate the mean 'mpg' for each group of 'origin' in the cars DataFrame.

In [94]:
# Obsolete dfply / only for reference


*summarize_each()*

The summarize_each() function is used to calculate summary statistics for groups of rows. It accepts keyword arguments of the form new_column_name = new_column_value, where new_column_value is a symbolic Series object.

### Exercise 19.

Calculate the mean 'mpg' and 'horsepower' for each group of 'origin' in the cars DataFrame.

In [95]:
# Obsolete dfply / only for reference


*summarize() can of course be used with groupings as well.*

### Exercise 20.

Calculate the mean 'mpg' for each group of 'origin' and 'model_year' in the cars DataFrame.

In [96]:
# Obsolete dfply / only for reference